# Level 2 - The tubule: chemistry at the membrane
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/kidney-sim/blob/main/notebooks/02_tubule_transport.ipynb)

**Goal:** follow the filtrate along the nephron and see how transporters and osmosis turn
180 L/day of filtrate into ~1.5 L of urine.

Here the **chemical** part of the model lives: pumps, co-transporters and **exchangers**.

In [ ]:
# --- Setup: run this cell first --------------------------------------------
import os, sys, subprocess
REPO_URL = "https://github.com/YOUR-USERNAME/kidney-sim.git"   # change if you forked the project
if "google.colab" in sys.modules and not os.path.exists("kidneysim"):
    if not os.path.exists("kidney-sim"):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("kidney-sim")
for path in [".", ".."]:
    if os.path.exists(os.path.join(path, "kidneysim")):
        sys.path.insert(0, os.path.abspath(path))
        break

import numpy as np
import matplotlib.pyplot as plt
import kidneysim as ks
from kidneysim import viz2d, viz3d
from kidneysim.ui import sliders
print("kidneysim", ks.__version__, "| interactive 3D (plotly):", viz3d.HAS_PLOTLY)

## 1. Governing equations along the tube (position $s$)

Flow of water $F$ and of each solute $N_i = F\,C_i$:

| process | equation | example |
|---|---|---|
| saturable transporter | $\dfrac{dN_i}{ds} = -J_{max}\dfrac{C_i}{K_m + C_i}$ | glucose (SGLT), salt (NKCC2, NCC) |
| leaky / proportional | $\dfrac{dN_i}{ds} = -k\,N_i$ | Na in proximal tubule, collecting duct (aldosterone) |
| diffusion (Fick) | $\dfrac{dN_u}{ds} = -P\,(C_{u,lumen}-C_{u,plasma})$ | urea |
| **exchanger** secretion | $\dfrac{dN_d}{ds} = +S_{max}\dfrac{C_{plasma}}{K_m + C_{plasma}}$ | drugs/toxins via OAT1 (swaps α-ketoglutarate for the toxin) |
| osmosis | $\dfrac{dF}{ds} = -L_p\,(Osm_{tissue}-Osm_{lumen})$ | water follows salt |

with $Osm_{lumen} = (2N_{Na} + N_{glucose} + N_{urea})/F$.
In the collecting duct the water permeability is set by the hormone **ADH**: $L_p = L_{p0} + L_{p,ADH}\cdot ADH$.

In [ ]:
r = ks.tubule.run_nephron()          # normal conditions
print(ks.tubule.summary(r))
viz2d.plot_nephron_profiles(r);

### 2D map and 3D nephron
The grey wall behind the nephron shows the **tissue** osmolarity: 290 in the cortex, up to 1200 mOsm/L deep in the medulla.
Change `quantity` to `"C_urea"`, `"C_Na"`, `"C_glucose"` or `"F"`.

In [ ]:
viz2d.plot_nephron_map(r, quantity="osm");

In [ ]:
viz3d.show(viz3d.nephron_3d(r, quantity="osm"))

## 2. Glucose: when transporters saturate (diabetes)
Above ~10–12 mmol/L of blood glucose, the SGLT transporters are full and glucose appears in the urine.

In [ ]:
g, filt, reab, exc = ks.tubule.glucose_titration()
viz2d.plot_glucose_titration(g, filt, reab, exc);

## 3. Experiment: hormones and drugs
* **ADH** (0 → 1): no ADH = lots of dilute urine; max ADH = little concentrated urine.
* **aldosterone**: more salt kept.
* **furosemide** blocks NKCC2 (ascending limb); **thiazide** blocks NCC (distal tubule); **SGLT2 inhibitor** blocks glucose reabsorption.

In [ ]:
def experiment(ADH=0.65, aldosterone=1.0, glucose=5.0, furosemide=0.0, thiazide=0.0, sglt2_inhibitor=0.0):
    r = ks.tubule.run_nephron(ADH=ADH, aldosterone=aldosterone, plasma={"glucose": glucose},
                              furosemide=furosemide, thiazide=thiazide, sglt2_inhibitor=sglt2_inhibitor)
    print(ks.tubule.summary(r))
    viz2d.plot_nephron_profiles(r)
    plt.show()

sliders(experiment, ADH=(0, 1, 0.05, 0.65), aldosterone=(0, 3, 0.1, 1.0), glucose=(3, 30, 1, 5),
        furosemide=(0, 1, 0.1, 0.0), thiazide=(0, 1, 0.1, 0.0), sglt2_inhibitor=(0, 1, 0.1, 0.0))

## Questions
1. Where is most of the water reabsorbed? And most of the salt?
2. Why does a person with uncontrolled diabetes pee a lot (*diabetes mellitus* = "sweet siphon")?
3. Why is the fluid at the end of the ascending limb *more dilute* than blood?
4. The drug clearance is close to the renal plasma flow (~600 mL/min). Which transport process explains this?

## What is missing?
We *assumed* a salt gradient in the medulla. Who builds it? → Level 3.